# Starfysh simulated spatial transcriptomics tutorial in scviva-tools

This notebook adapts the upstream Starfysh simulation tutorial to the `scviva.external.Starfysh` Phase 2A API.

The active cells run expression-only deconvolution and expose proportions, latent representation, and model outputs. Histology PoE, archetypal analysis, and cell-type-specific expression are kept as future-capability notes.


In [ ]:
!pip install --quiet scviva-tools


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch

import scviva
from scviva.external import Starfysh

scviva.settings.seed = 0
torch.manual_seed(0)


In [ ]:
def _as_dense(x):
    if hasattr(x, "toarray"):
        return x.toarray()
    return np.asarray(x)


def compute_signature_scores(adata, gene_signatures):
    """Compute simple per-spot signature priors from marker-gene columns.

    The current scviva Starfysh wrapper expects one prior score per spot and
    cell type. Upstream Starfysh notebooks often start from marker-gene lists;
    this helper turns those marker lists into normalized spot-level priors.
    """
    signatures = gene_signatures.copy()
    if "Unnamed: 0" in signatures.columns:
        signatures = signatures.drop(columns=["Unnamed: 0"])

    scores = pd.DataFrame(index=adata.obs_names)
    for cell_type in signatures.columns:
        markers = signatures[cell_type].dropna().astype(str)
        markers = [gene for gene in markers if gene in adata.var_names]
        if len(markers) == 0:
            scores[cell_type] = 0.0
            continue
        values = _as_dense(adata[:, markers].layers.get("counts", adata[:, markers].X))
        scores[cell_type] = values.mean(axis=1)

    scores = scores.clip(lower=0)
    row_sums = scores.sum(axis=1).replace(0, np.nan)
    return scores.div(row_sums, axis=0).fillna(1.0 / scores.shape[1])


## Load simulated data and marker signatures

The upstream notebook uses Starfysh's `utils.load_adata`. In scviva-tools, load or prepare an AnnData directly, then pass counts and spatial coordinates through `Starfysh.setup_anndata`.


In [ ]:
data_path = Path("../data")
sample_id = "simulated_ST_data_1"

candidate_h5ads = [
    data_path / sample_id / "st.h5ad",
    data_path / sample_id / "adata.h5ad",
    data_path / f"{sample_id}.h5ad",
]
for adata_path in candidate_h5ads:
    if adata_path.exists():
        break
else:
    raise FileNotFoundError(
        "Could not find a simulated AnnData file. Update candidate_h5ads for your local data."
    )

adata = sc.read_h5ad(adata_path)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

if "spatial" not in adata.obsm:
    coords_path = data_path / sample_id / "spot_list.csv"
    if not coords_path.exists():
        raise FileNotFoundError(
            f"Spatial coordinates not found at {coords_path}. "
            "Provide a CSV with spot barcodes as index and two coordinate columns."
        )
    coords = pd.read_csv(coords_path, index_col=0).reindex(adata.obs_names)
    adata.obsm["spatial"] = coords.iloc[:, :2].to_numpy(dtype=np.float32)

gene_sig = pd.read_csv(data_path / "tnbc_signature.csv")
signature_scores = compute_signature_scores(adata, gene_sig)
signature_scores.head()


## Run Starfysh expression-only deconvolution


> **Note: raw counts required.** The Starfysh model expects raw integer count data in the registered layer (default: `adata.X`). Do **not** log-normalize or scale before calling `setup_anndata` — the model applies its own library-size normalization internally. Normalizing beforehand will silently degrade training.

In [ ]:
Starfysh.setup_anndata(adata, layer="counts", spatial_key="spatial")
model = Starfysh(
    adata,
    signature_scores=signature_scores,
    n_latent=10,
    n_hidden=128,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.train(max_epochs=100, batch_size=128, lr=1e-3, device=device, prog_bar=True)


In [ ]:
proportions = model.get_proportions(batch_size=128, store_key="starfysh_proportions")
latent = model.get_latent_representation(batch_size=128, store_key="X_starfysh")
outputs = model.get_model_outputs(batch_size=128, store=True)

print("proportions:", proportions.shape)
print("latent:", latent.shape)
print("model outputs:", {k: v.shape for k, v in outputs.items()})
proportions.head()


In [ ]:
for cell_type in proportions.columns[: min(4, proportions.shape[1])]:
    adata.obs[f"starfysh_{cell_type}"] = proportions[cell_type].values

sc.pp.neighbors(adata, use_rep="X_starfysh")
sc.tl.umap(adata)
sc.pl.umap(
    adata,
    color=[f"starfysh_{ct}" for ct in proportions.columns[: min(4, proportions.shape[1])]],
    frameon=False,
    ncols=2,
)


## Deferred upstream sections


In [ ]:
# TODO: Phase 4 — Starfysh PoE and histology integration
# Upstream reference only:
#   model, loss = utils.run_starfysh(visium_args, poe=True, ...)
#
# TODO: Phase 6 — archetypal analysis and anchor refinement
# Upstream reference only:
#   aa_model = AA.ArchetypalAnalysis(adata_orig=adata)
#   visium_args = utils.refine_anchors(visium_args, aa_model, ...)
#
# TODO: Later Phase 2/6 — cell-type-specific inferred expression and plotting
# Upstream reference only:
#   pred_exprs = sf_model.model_ct_exp(model, adata, visium_args, device=device)
